# RG3 열 모델 재현 및 점진 축소


In [1]:
from pathlib import Path
import json,itertools,nbformat
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import NearestNeighbors
ROOT=Path.cwd();OUT=ROOT/'reports/tables'
source54=''.join(c.source for c in nbformat.read('notebooks/54_rg3_pdf_thermal_confirm_gates.ipynb',4).cells if c.cell_type=='code')
exec(source54.split('\nrecords=[]')[0])
headnames=['RF42','RF7','RF101','ET42','ET7','ET101','LR01','LR1','LR10']
sets={'thermal9':headnames,'thermal7':headnames[:6]+['LR1'],'thermal6':headnames[:6],'thermal3':['RF42','ET42','LR1'],'thermal2':['RF42','ET42']}
cache={};audit=[]
def heads(ref,q):
 key=(tuple(ref),tuple(q))
 if key in cache:return cache[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 audit.append(dict(train=len(ref),query=len(q),overlap=len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q]))))
 if len(np.unique(y[ref]))<2:return np.zeros((len(q),9))
 tr=local_residual(ref,ref,'surface_thermal_contact');te=local_residual(ref,q,'surface_thermal_contact')
 imp=SimpleImputer(keep_empty_features=True).fit(tr);tr=imp.transform(tr);te=imp.transform(te)
 out=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  for seed in [42,7,101]:
   m=cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2).fit(tr,y[ref])
   out.append(m.predict_proba(te)[:,1])
 sc=RobustScaler().fit(tr[y[ref]==0])
 for C in [.1,1.,10.]:
  m=LogisticRegression(C=C,class_weight='balanced',max_iter=2000,random_state=42).fit(sc.transform(tr),y[ref])
  out.append(m.predict_proba(sc.transform(te))[:,1])
 cache[key]=np.column_stack(out);return cache[key]
def score(a,names):
 return np.mean([a[:,[headnames.index(n) for n in names if n.startswith(prefix)]].mean(axis=1) for prefix in ['RF','ET','LR'] if any(n.startswith(prefix) for n in names)],axis=0)
expert=pd.read_csv(OUT/'rg3_innovation_seed_sensitivity_predictions.csv')
expert=expert[expert.scheme.eq('leave_one_block_out')]
pressure=pd.read_csv(OUT/'rg3_191_pressure_head_reduction_predictions.csv')
pressure=pressure[pressure.candidate.eq('5_one_seed_each')].set_index('TimeStamp')
oldthermal=pd.read_csv(OUT/'rg3_pdf_confirm_gates_predictions.csv')
oldthermal=oldthermal[oldthermal.scheme.eq('leave_one_block_out')&oldthermal.model.eq('thermal_change_low_0.05')].set_index('TimeStamp')
records=[];components=[]
for held in blocks:
 q=np.flatnonzero(groups.eq(held));ref=np.flatnonzero(~groups.eq(held))
 outer=heads(ref,q);inner=np.full((len(ref),9),np.nan);zi=np.full((len(ref),6),np.nan)
 zo=local_residual(ref,q,'surface_thermal_contact')
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy()
  inner[mask]=heads(ref[~mask],ref[mask])
  zi[mask]=local_residual(ref[~mask],ref[mask],'surface_thermal_contact')
 gate=float(np.quantile(zi[y[ref]==0,4],.05))
 times=d.TimeStamp.iloc[q].astype(str)
 bpressure=pressure.loc[times,'prediction'].to_numpy(bool)
 for tname,tnames in sets.items():
  si=score(inner,tnames);so=score(outer,tnames)
  cut=float(np.quantile(si[y[ref]==0],.975,method='higher'))
  th=(so>cut)&(zo[:,4]<gate)
  if tname=='thermal9':
   orig=oldthermal.loc[times]
   assert np.allclose(so,orig.score,atol=1e-12) and np.allclose(cut,orig.score_threshold,atol=1e-12)
   assert np.array_equal(th,(orig.score.to_numpy()>orig.score_threshold.to_numpy()) & orig.gate_pass.to_numpy(bool))
  for ename,ecount in [('three_seed_mean',12),('seed42',4),('seed7',4),('seed101',4),('none',0)]:
   if ename=='none':ev=np.zeros(len(q),bool)
   else:
    ex=expert[expert.model.eq(ename)].set_index('TimeStamp').loc[times]
    ev=(ex.expert_score.to_numpy()>ex.expert_threshold.to_numpy())&(ex.current_barrel5_delta.to_numpy()>ex.delta_threshold.to_numpy())
   pr=bpressure|ev|th
   label=ename+'+'+tname
   for j,idx in enumerate(q):records.append(dict(candidate=label,classifiers=5+ecount+len(tnames),TimeStamp=d.TimeStamp.iloc[idx],block=held,actual=int(y[idx]),prediction=int(pr[j])))
  for j,idx in enumerate(q):components.append(dict(thermal=tname,TimeStamp=d.TimeStamp.iloc[idx],score=float(so[j]),cutoff=cut,gate=gate,prediction=int(th[j])))
 pd.DataFrame(records).to_csv(OUT/'rg3_192_thermal_expert_reduction_progress.csv',index=False)
 print('Thermal refit verified',held,flush=True)
pp=pd.DataFrame(records);mets=[]
for label,g in pp.groupby('candidate',sort=False):
 assert len(g)==583 and g.TimeStamp.is_unique
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 mets.append(dict(candidate=label,classifiers=int(g.classifiers.iloc[0]),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=float(2*tp/(2*tp+fp+fn))))
mm=pd.DataFrame(mets)
pp.to_csv(OUT/'rg3_192_thermal_expert_reduction_predictions.csv',index=False,encoding='utf-8-sig')
mm.to_csv(OUT/'rg3_192_thermal_expert_reduction_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(components).to_csv(OUT/'rg3_192_thermal_components.csv',index=False)
pd.DataFrame(audit).to_csv(OUT/'rg3_192_thermal_fit_audit.csv',index=False)
assert all(a['overlap']==0 for a in audit)
print(mm.sort_values(['TP','FP','classifiers'],ascending=[False,True,True]).to_string(index=False))
text='# RG3 26개에서 추가 축소\n\n동일583샷(정상566/가스17),5구간 제외 개발 비교. 기존열9개를실제로재학습해각구간의원본점수/임계값/단독예측모두재현한후열반복을축소했다. 압력은191에서재학습한단일RF/ET/LR3개,기본RF2예측과결합한것을사용했다. 패턴전문가12개→단일seed4개비교는47번의기존구간제외점수및그seed의학습내부임계값을사용했다(이번에전문가를새로학습하지않음).\n\n'
text+=mm.sort_values(['classifiers','TP','FP'],ascending=[True,False,True]).to_string(index=False)
text+='\n\n## 한계\n독립최종검증이아니며평가결과를본축소후보선택이다. 전체조합을빈환경에서처음부터재학습한검증은아직아니다. 기존이력피처조건재사용,비라벨경보율미확인,모델파일미교체,commit/push없음. 분류기수에Ridge4전처리회귀는별도다.\n'
(ROOT/'reports/rg3_192_thermal_expert_reduction.md').write_text(text,encoding='utf-8')



Feature group surface_thermal_contact ['barrel_mold_gap', 'mold_side_difference', 'mold_delta_mean', 'mold_dev_mean', 'barrel_mold_delta_gap', 'pressure_per_filling_time']


C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\nbformat\validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


Thermal refit verified 20201021_001626_RG3


Thermal refit verified 20201022_002405_RG3


Thermal refit verified 20201022_044026_RG3


Thermal refit verified 20201023_010128_RG3


Thermal refit verified 20201023_044148_RG3


               candidate  classifiers  TP  FP  FN  TN       F1
         seed42+thermal7           16   8   6   9 560 0.516129
        seed101+thermal7           16   8   6   9 560 0.516129
         seed42+thermal9           18   8   6   9 560 0.516129
        seed101+thermal9           18   8   6   9 560 0.516129
three_seed_mean+thermal7           24   8   6   9 560 0.516129
three_seed_mean+thermal9           26   8   6   9 560 0.516129
         seed42+thermal3           12   8   7   9 559 0.500000
        seed101+thermal3           12   8   7   9 559 0.500000
three_seed_mean+thermal3           20   8   7   9 559 0.500000
         seed42+thermal2           11   7   6  10 560 0.466667
        seed101+thermal2           11   7   6  10 560 0.466667
           none+thermal7           12   7   6  10 560 0.466667
           none+thermal9           14   7   6  10 560 0.466667
         seed42+thermal6           15   7   6  10 560 0.466667
        seed101+thermal6           15   7   6  10 560 0

1979